# 🟢 02 — Giving the Agent Eyes: Video Captioning

> **Goal:** show the agent a video and have it describe what happens.

---

## How does a computer "watch" a video?

It doesn't watch like we do. It takes **snapshots** (frames) a few times per second, turns
each into number-patterns it understands, then reasons over them.

```mermaid
flowchart LR
    VID["video<br/>scene.mp4"]:::data
    FR["grab frames<br/>(e.g. 4 per second)"]:::data
    ENC["turn each frame<br/>into vision tokens"]:::tool
    BRAIN["CosmosVisionModel<br/>looks + thinks"]:::reason
    CAP(["Two workers walk<br/>past a yellow loader…"]):::out
    VID --> FR --> ENC --> BRAIN --> CAP
    classDef user fill:#ECEFF1,stroke:#607D8B,stroke-width:2px,color:#263238
    classDef agent fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
    classDef reason fill:#E8F5E9,stroke:#388E3C,stroke-width:2px,color:#1B5E20
    classDef gen fill:#F3E5F5,stroke:#8E24AA,stroke-width:2px,color:#4A148C
    classDef tool fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#E65100
    classDef data fill:#FFFDE7,stroke:#FBC02D,stroke-width:2px,color:#F57F17
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
    classDef think fill:#FCE4EC,stroke:#D81B60,stroke-width:2px,color:#880E4F
```

The knob that controls *how many snapshots* is **`fps`** (frames per second). More frames =
more detail but slower. `fps=4` is a great default.

In [ ]:
# 🔧 Preflight — this cell never crashes. It just tells us what's available.
import os, sys, time
os.environ.setdefault("QWEN_VL_VIDEO_READER", "torchcodec")
sys.path.insert(0, os.path.abspath(".."))

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety.
# The notebooks live in notebooks/ but our sample media sits one level up, so we
# explicitly widen the workspace to the project root + /tmp. (This is how you grant
# the tools access to a folder — never wider than you need.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])            # import strands_cosmos from repo root

PROJECT_ROOT = os.path.abspath("..")
SAMPLE_VIDEO = os.path.join(PROJECT_ROOT, "sample.mp4")
SAMPLE_IMAGE = os.path.join(PROJECT_ROOT, "sample.png")

def gpu_ready() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

READY = gpu_ready()
print("🎮 GPU available :", READY)
if READY:
    import torch
    print("   Device       :", torch.cuda.get_device_name(0))
    print(f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("   (CPU-only is fine for reading along — the heavy cells will skip safely.)")
print("📹 sample.mp4   :", os.path.exists(SAMPLE_VIDEO))
print("🖼  sample.png   :", os.path.exists(SAMPLE_IMAGE))

## Step 1 — Peek at the video first (no AI needed)
The `video_probe` tool wraps `ffprobe` to read size, resolution, and duration. Tools like
this work **without a GPU**.

In [ ]:
from strands_cosmos import video_probe
if os.path.exists(SAMPLE_VIDEO):
    info = video_probe(video_path=SAMPLE_VIDEO)
    print(info["content"][0]["text"][:400])
else:
    print("⏭  sample.mp4 not found next to the repo root.")

## Step 2 — Load the vision model
`CosmosVisionModel` is the text brain **plus eyes**. The `fps` and token limits keep long
videos from running out of memory.

In [ ]:
agent = None
if READY and os.path.exists(SAMPLE_VIDEO):
    from strands import Agent
    from strands_cosmos import CosmosVisionModel
    t0 = time.time()
    model = CosmosVisionModel(
        model_id="nvidia/Cosmos-Reason2-2B",
        fps=4,                                   # 4 snapshots per second
        params={"max_tokens": 1024, "temperature": 0.6},
    )
    agent = Agent(model=model)
    print(f"✅ Vision agent ready in {time.time()-t0:.1f}s")
else:
    print("⏭  Need a GPU + sample.mp4 to run the model.")

## Step 3 — Caption the video
The magic is the **`<video>...</video>` tag**. Put the file path inside it and the model
knows to *look*, not just read.

In [ ]:
if agent is not None:
    t0 = time.time()
    agent(f"Caption in detail: <video>{SAMPLE_VIDEO}</video>")
    print(f"\n⏱  {time.time()-t0:.1f}s")
else:
    print("⏭  Skipped (no GPU / no video).")

## Step 4 — Ask a *specific* question about it
Same tag, different question. You're talking to the video now.

In [ ]:
if agent is not None:
    agent(f"<video>{SAMPLE_VIDEO}</video> How many distinct scenes or shot changes are there?")
else:
    print("⏭  Skipped.")

# 🎓 What you learned

| Concept | Takeaway |
|---------|----------|
| `CosmosVisionModel` | Text brain **+ eyes** (video & image) |
| `<video>path</video>` | The tag that attaches media to your question |
| `fps` | How many frames/sec to sample (detail vs. speed) |
| `video_probe` | Inspect a video with **no GPU** |

**Next:** [03 — Driving Analysis](03_driving_analysis.ipynb) — watch the agent *think step
by step*. 🧠 →